# confusion matrix — Python demo

Numerical companion to the entry [confusion matrix](https://dictionaryofml.org/terms/cm.html) of the [Dictionary of Applied Machine Learning](https://dictionaryofml.org/): it recomputes what the entry states and prints one line per check.

Numerical companion to the glossary entry 'cm' (confusion matrix). The GeoSphere Austria weather station Krems (station id 3805, 48.42 N, 15.62 E) records the air temperature every hour; this script downloads the records for March and December 2024 from the GeoSphere data hub (resource klima-v2-1h), keeps the reading at 06:00 (UTC) and the one at 15:00 (UTC) of each day, and writes them to cm_weather.csv, so the exact numbers behind every figure stay on record. Checks pin the downloaded values to the 2024 archive, so a change on the server side is caught rather than silently absorbed.

Requires NumPy and Matplotlib only, and uses fixed seeds, so the printed numbers reproduce exactly. Generated from [`pythondemos/cm.py`](https://dictionaryofml.org/terms/cm.py); CC BY 4.0.

Before running a downloaded copy, scan it: `uvx bandit -q -t B102,B307,B602,B605 cm.py` (B102,B307,B602,B605 are the exec, eval and shell checks of [bandit](https://bandit.readthedocs.io/)) reports nothing: the script runs no shell command and evaluates no generated code.

In [ ]:
# Notebook shim: the script resolves output paths relative to __file__,
# which a notebook kernel does not define; everything lands in the
# working directory instead.
import os
__file__ = os.path.join(os.getcwd(), "cm.py")
os.makedirs("pythondemos", exist_ok=True)

In [ ]:
"""Frost warning at Krems an der Donau: two classifiers with the same
accuracy, told apart only by their confusion matrices.

Purpose
-------
Numerical companion to the glossary entry 'cm' (confusion matrix).  The
GeoSphere Austria weather station Krems (station id 3805, 48.42 N,
15.62 E) records the air temperature every hour; this script downloads
the records for March and December 2024 from the GeoSphere data hub
(resource klima-v2-1h), keeps the reading at 06:00 (UTC) and the one at
15:00 (UTC) of each day, and writes them to cm_weather.csv, so the exact
numbers behind every figure stay on record.  Checks pin the downloaded values to the 2024 archive, so a
change on the server side is caught rather than silently absorbed.

Each of the 40 data points is one day: its features are the two
readings of that day, its label is whether the 06:00 reading of the
FOLLOWING day stays above 0 degrees C.  34 of the 40 following days stay
above 0 degrees C and only 6 bring frost, so the constant
prediction "above 0" -- a baseline that ignores the features --
is correct on 34 of the 40 days: accuracy 0.85.  A linear classifier
learned by logistic regression from the 40 data points reaches the same
accuracy 0.85.  The two confusion matrices nevertheless differ: the
learned classifier detects one of the six frost days at the cost of one
false alarm, while the baseline detects none.

The two 20-day windows (1-20 March and 5-24 December 2024, each day
paired with the following day) are chosen so that six of the forty
following days bring a frost morning: one in March, five in December.
A frost morning is rare in a year at this station -- the 06:00 reading
falls to 0 degrees or below on few days -- which is what makes accuracy
a poor summary here.

Deterministic: no randomness (the minimization starts from the zero
vector).  Self-contained: numpy + matplotlib only (stdlib urllib for
the download).

Blocks
------
[B-fetch] Download the hourly air temperature at Krems for March and
          December 2024 from the GeoSphere data hub, keep the 06:00 and
          the 15:00 (UTC) reading of each day; write the 62 records to
          cm_weather.csv and check them against the archive.
[B-data]  Build the 40 data points from the downloaded readings;
          check the label counts: 34 next days above 0 degrees C,
          6 with frost.
[B-learn] Learn a linear classifier from the 40 data points by logistic
          regression; check that the minimization has converged.
[B-cm]    Confusion matrices of the learned classifier and of the
          always-above-0 baseline; check that both reach accuracy 0.85
          and that only the confusion matrices distinguish them.  With
          frost as the positive class, the learned classifier has recall
          1/6 and precision 1/2, while the baseline has recall 0 and no
          precision at all: it predicts frost for no day, so the column
          that precision normalizes is empty.

Outputs
-------
cm_weather.csv      : date, t06, t15 of the 62 downloaded days
cm_points_above.csv : t06, t15 of the 34 days followed by a day above 0
cm_points_frost.csv : t06, t15 of the 6 days followed by a frost morning
cm_boundary.csv     : t06, t15 along the learned decision boundary
cm_counts.csv       : the four entries of both confusion matrices
cm.png              : preview (checking only) -- the 40 data points with
                      the decision boundary, and the two confusion
                      matrices
"""

# /// script
# requires-python = ">=3.10"
# dependencies = ["matplotlib==3.10.7", "numpy==2.3.4"]
# ///

import datetime
import json
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = Path(__file__).parent

report = []                         # collects (check name, pass/fail) pairs


def check(name, ok):                # records and prints one verification
    report.append((name, bool(ok)))
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")

**[B-fetch]** Download the hourly air temperature at Krems for March and December 2024 from the GeoSphere data hub, keep the 06:00 and the 15:00 (UTC) reading of each day; write the 62 records to cm_weather.csv and check them against the archive.

In [ ]:
URL = ("https://dataset.api.hub.geosphere.at/v1/station/historical/"
       "klima-v2-1h?parameters=tl&station_ids=3805"
       "&start=2024-03-01T00:00&end=2024-12-31T23:00")
with urllib.request.urlopen(URL, timeout=300) as resp:
    payload = json.load(resp)
readings = payload["features"][0]["properties"]["parameters"]["tl"]["data"]
byday = {}
for stamp, value in zip(payload["timestamps"], readings):
    if value is None:                           # the archive drops whole days
        continue
    hour = stamp[11:16]                         # the stamps are UTC, and so
    if hour in ("06:00", "15:00"):              # are the two times
        byday.setdefault(stamp[:10], {})[hour] = float(value)
records = [(day, rec["06:00"], rec["15:00"])
           for day, rec in sorted(byday.items())
           if len(rec) == 2 and day[5:7] in ("03", "12")]   # March, December
with open(OUT_DIR / "cm_weather.csv", "w") as f:
    f.write("date,t06,t15\n")
    for day, lo, hi in records:
        f.write(f"{day},{lo},{hi}\n")
check("[B-fetch] 62 days downloaded for March and December 2024",
      len(records) == 62)
check("[B-fetch] the record matches the archive (1 March: 5.4 at 06:00 "
      "and 9.3 at 15:00, both UTC)",
      records[0] == ("2024-03-01", 5.4, 9.3))

**[B-data]** Build the 40 data points from the downloaded readings; check the label counts: 34 next days above 0 degrees C, 6 with frost.

In [ ]:
dates = [day for day, _, _ in records]
temps = np.array([[lo, hi] for _, lo, hi in records])
date_index = {d: i for i, d in enumerate(dates)}

predictor_days = ([date_index["2024-03-01"] + k for k in range(20)] +
                  [date_index["2024-12-05"] + k for k in range(20)])
idx = np.array(predictor_days)
check("[B-data] every day of the 40 is followed in the record by the next "
      "calendar day",
      all(datetime.date.fromisoformat(dates[i + 1])
          - datetime.date.fromisoformat(dates[i]) == datetime.timedelta(1)
          for i in idx))

X = temps[idx]                            # features: (t06, t15) of day t
y = np.where(temps[idx + 1, 0] > 0.0, 1.0, -1.0)   # +1: next 06:00 above 0
above, frost = y > 0, y < 0

check("[B-data] 40 data points, 20 per window", len(y) == 40)
check("[B-data] 34 next mornings stay above 0 degrees C",
      int(above.sum()) == 34)
check("[B-data] 6 next mornings bring frost, 1 in March and 5 in December",
      int(frost.sum()) == 6 and int(frost[:20].sum()) == 1)

**[B-learn]** Learn a linear classifier from the 40 data points by logistic regression; check that the minimization has converged.

In [ ]:
Xb = np.hstack([X, np.ones((len(y), 1))])          # append constant feature
w = np.zeros(3)
STEP, ITERS = 0.01, 200_000
for _ in range(ITERS):
    grad = -(y[:, None] * Xb / (1 + np.exp(y * (Xb @ w)))[:, None]).mean(0)
    w -= STEP * grad
check("[B-learn] the minimization has converged (tiny final update)",
      float(np.linalg.norm(grad)) < 1e-4)
print(f"  learned weights: {w[0]:+.3f} * t06 {w[1]:+.3f} * t15 {w[2]:+.3f}")

pred = np.where(Xb @ w > 0, 1.0, -1.0)
pred_baseline = np.ones(len(y))                    # always "above 0"


def confusion(y_true, y_pred):
    """2x2 counts; rows: true above 0 / frost, columns: predicted."""
    return np.array([[int(np.sum((y_true == a) & (y_pred == p)))
                      for p in (1.0, -1.0)] for a in (1.0, -1.0)])


cm_clf = confusion(y, pred)
cm_base = confusion(y, pred_baseline)
acc_clf = float(np.trace(cm_clf)) / len(y)
acc_base = float(np.trace(cm_base)) / len(y)

**[B-cm]** Confusion matrices of the learned classifier and of the always-above-0 baseline; check that both reach accuracy 0.85 and that only the confusion matrices distinguish them. With frost as the positive class, the learned classifier has recall 1/6 and precision 1/2, while the baseline has recall 0 and no precision at all: it predicts frost for no day, so the column that precision normalizes is empty.

In [ ]:
print(f"  learned classifier: {cm_clf.tolist()}  accuracy {acc_clf:.2f}")
print(f"  baseline:           {cm_base.tolist()}  accuracy {acc_base:.2f}")
check("[B-cm] both reach accuracy 0.85",
      abs(acc_clf - 0.85) < 1e-9 and abs(acc_base - 0.85) < 1e-9)
check("[B-cm] the learned classifier detects a frost day, one false alarm",
      cm_clf[1, 1] == 1 and cm_clf[0, 1] == 1)
check("[B-cm] the baseline detects no frost day at all", cm_base[1, 1] == 0)

# frost is the positive class: recall normalizes its ROW, precision its COLUMN
rec_clf = cm_clf[1, 1] / cm_clf[1].sum()
prec_den = cm_clf[:, 1].sum()
prec_clf = cm_clf[1, 1] / prec_den
print(f"  frost as the positive class: recall {cm_clf[1, 1]}/{cm_clf[1].sum()}"
      f" = {rec_clf:.4f}, precision {cm_clf[1, 1]}/{prec_den} = {prec_clf:.4f}")
check("[B-cm] with frost as the positive class the learned classifier has "
      "recall 1/6 and precision 1/2",
      abs(rec_clf - 1 / 6) < 1e-12 and abs(prec_clf - 1 / 2) < 1e-12)
check("[B-cm] the baseline has recall 0, and no precision at all: it makes no "
      "frost prediction, so the column precision would normalize is empty",
      cm_base[1, 1] / cm_base[1].sum() == 0.0 and cm_base[:, 1].sum() == 0)

# ---- outputs: CSVs for the entry's pgfplots figure, preview PNG
header = "t06,t15"
np.savetxt(OUT_DIR / "cm_points_above.csv", X[above], delimiter=",",
           header=header, comments="", fmt="%.1f")
np.savetxt(OUT_DIR / "cm_points_frost.csv", X[frost], delimiter=",",
           header=header, comments="", fmt="%.1f")
t_line = np.linspace(-4.5, 9.0, 2)
boundary = np.stack([t_line, -(w[0] * t_line + w[2]) / w[1]], 1)
np.savetxt(OUT_DIR / "cm_boundary.csv", boundary, delimiter=",",
           header=header, comments="", fmt="%.3f")
with open(OUT_DIR / "cm_counts.csv", "w") as f:
    f.write("classifier,true,pred_above,pred_frost\n")
    for name, cm in (("learned", cm_clf), ("baseline", cm_base)):
        for row, true in zip(cm, ("above", "frost")):
            f.write(f"{name},{true},{row[0]},{row[1]}\n")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4),
                         gridspec_kw={"width_ratios": [1.6, 1, 1]})
ax = axes[0]
ax.plot(X[above, 0], X[above, 1], "o", color="tab:blue", ms=5,
        label="next day above 0 °C")
ax.plot(X[frost, 0], X[frost, 1], "^", mfc="none", mec="tab:red", ms=8,
        mew=1.5, label="next day frost")
ax.plot(boundary[:, 0], boundary[:, 1], "k--", lw=1.2,
        label="decision boundary")
ax.set_xlabel("06:00 (UTC) reading of the day (°C)")
ax.set_ylabel("15:00 (UTC) reading of the day (°C)")
ax.set_ylim(0, 19)
ax.set_title("40 days at Krems, March/December 2024")
ax.legend(frameon=False, fontsize=8)

for ax, name, cm, acc in ((axes[1], "learned classifier", cm_clf, acc_clf),
                          (axes[2], "always-above-0 baseline", cm_base,
                           acc_base)):
    ax.imshow(cm, cmap="Greys", vmin=0, vmax=45)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="black" if cm[i, j] < 25 else "white")
    ax.set_xticks([0, 1], ["above 0", "frost"])
    ax.set_yticks([0, 1], ["above 0", "frost"])
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(f"{name}\naccuracy {acc:.2f}")

fig.tight_layout()
fig.savefig(OUT_DIR / "cm.png", dpi=150)

failed = [name for name, ok in report if not ok]
print(f"{len(report) - len(failed)}/{len(report)} checks passed"
      + (f", FAILED: {failed}" if failed else ""))